In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 02 ? SAR Preprocessing + Terrain + Multi-Hazard Features
## Preprocesses Sentinel-1 scenes and builds flood, erosion, and landslide feature manifests


## Section 2.1 — Environment Setup

Listing the available scenes and standardizing paths up front prevents silent failures later in the preprocessing chain. It also confirms that the raw DEM and JRC inputs needed by the HTC branch are already available locally.


In [ ]:
import json
import os
import subprocess
from datetime import datetime
from pathlib import Path
from xml.etree.ElementTree import Element, SubElement, tostring

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from dotenv import load_dotenv
from pyproj import Transformer
from rasterio.enums import Resampling
from rasterio.warp import reproject
from tqdm.auto import tqdm

RAW_SENTINEL = RAW_DIR / 'sentinel1'
RAW_INFRA = RAW_DIR / 'infrastructure'
RAW_DEM = RAW_DIR / 'dem'
PROCESSED_SAR = PROCESSED_DIR / 'sar'
PROCESSED_TERRAIN = PROCESSED_DIR / 'terrain'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORT_DIR = OUTPUTS_DIR / 'report'
for folder in [PROCESSED_SAR, PROCESSED_TERRAIN, FIGURES_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / '.env')
SNAP_GPT_PATH = os.getenv('SNAP_GPT_PATH', 'gpt')
EPSG_TARGET = 'EPSG:32646'
AOI_BOUNDS = (91.6, 24.5, 92.5, 25.2)
SAFE_SCENES = sorted(list(RAW_SENTINEL.glob('*.SAFE'))) + sorted(list(RAW_SENTINEL.glob('*.zip')))
print(f'Detected {len(SAFE_SCENES)} scene containers')
for scene in SAFE_SCENES:
    print(' -', scene.name)


## Section 2.2 — SAR Preprocessing Chain

The preprocessing function applies orbit correction, calibration, speckle filtering, multilooking, terrain correction, AOI clipping, and VV/VH export. Keeping this logic in a single function ensures every scene is treated consistently before training or inference.


In [ ]:
def scene_date_from_name(scene_path: Path) -> str:
    tokens = scene_path.name.replace('.SAFE', '').replace('.zip', '').split('_')
    for token in tokens:
        if token.isdigit() and len(token) >= 8:
            return token[:8]
    return datetime.utcnow().strftime('%Y%m%d')

def build_snap_graph(scene_path: Path, dem_path: Path, output_path: Path) -> str:
    graph = Element('graph', id='Graph')
    SubElement(graph, 'version').text = '1.0'

    read = SubElement(graph, 'node', id='Read')
    SubElement(read, 'operator').text = 'Read'
    params = SubElement(read, 'parameters')
    SubElement(params, 'file').text = str(scene_path)

    prev = 'Read'
    for node_id, op_name, extras in [
        ('Orbit', 'Apply-Orbit-File', {}),
        ('Calib', 'Calibration', {'outputSigmaBand': 'true', 'selectedPolarisations': 'VV,VH'}),
        ('Speckle', 'Speckle-Filter', {'filter': 'Lee Sigma', 'filterSizeX': '7', 'filterSizeY': '7'}),
        ('Multilook', 'Multilook', {'nRgLooks': '2', 'nAzLooks': '2'}),
        ('Terrain', 'Terrain-Correction', {'demName': 'External DEM', 'externalDEMFile': str(dem_path), 'mapProjection': EPSG_TARGET, 'pixelSpacingInMeter': '30'}),
    ]:
        node = SubElement(graph, 'node', id=node_id)
        SubElement(node, 'operator').text = op_name
        sources = SubElement(node, 'sources')
        SubElement(sources, 'sourceProduct', refid=prev)
        node_params = SubElement(node, 'parameters')
        for key, value in extras.items():
            SubElement(node_params, key).text = value
        prev = node_id

    write = SubElement(graph, 'node', id='Write')
    SubElement(write, 'operator').text = 'Write'
    sources = SubElement(write, 'sources')
    SubElement(sources, 'sourceProduct', refid=prev)
    params = SubElement(write, 'parameters')
    SubElement(params, 'file').text = str(output_path)
    SubElement(params, 'formatName').text = 'GeoTIFF-BigTIFF'
    return tostring(graph, encoding='unicode')

def preprocess_scene(scene_path: Path, output_dir: Path) -> dict:
    import tempfile
    scene_date = scene_date_from_name(scene_path)
    temp_out = output_dir / f'S1_stack_{scene_date}.tif'
    graph_xml = build_snap_graph(scene_path, RAW_DEM / 'srtm_sylhet_30m.tif', temp_out)
    with tempfile.NamedTemporaryFile('w', suffix='.xml', delete=False) as tmp:
        tmp.write(graph_xml)
        graph_path = Path(tmp.name)
    try:
        subprocess.run([SNAP_GPT_PATH, str(graph_path)], check=True, capture_output=True, text=True)
    finally:
        graph_path.unlink(missing_ok=True)

    with rasterio.open(temp_out) as src:
        bounds = rasterio.warp.transform_bounds('EPSG:4326', src.crs, *AOI_BOUNDS)
        window = rasterio.windows.from_bounds(*bounds, transform=src.transform)
        vv = src.read(1, window=window).astype(np.float32)
        vh = src.read(2, window=window).astype(np.float32)
        transform = rasterio.windows.transform(window, src.transform)
        profile = src.profile.copy()
        profile.update(count=1, dtype='float32', transform=transform, width=vv.shape[1], height=vv.shape[0])

    vv_db = 10 * np.log10(np.clip(vv, 1e-6, None))
    vh_db = 10 * np.log10(np.clip(vh, 1e-6, None))
    vv_path = output_dir / f'S1_VV_{scene_date}.tif'
    vh_path = output_dir / f'S1_VH_{scene_date}.tif'
    with rasterio.open(vv_path, 'w', **profile) as dst:
        dst.write(vv_db, 1)
    with rasterio.open(vh_path, 'w', **profile) as dst:
        dst.write(vh_db, 1)
    temp_out.unlink(missing_ok=True)
    return {'date': scene_date, 'vv_path': str(vv_path), 'vh_path': str(vh_path), 'status': 'success'}


## Section 2.3 — Batch SAR Processing

Running the chain over every discovered scene creates a consistent processing log and gives later notebooks a stable set of VV/VH rasters to reference by date.


In [ ]:
log_rows = []
for scene_path in tqdm(SAFE_SCENES, desc='Preprocessing scenes'):
    try:
        log_rows.append(preprocess_scene(scene_path, PROCESSED_SAR))
    except Exception as exc:
        log_rows.append({'date': scene_date_from_name(scene_path), 'vv_path': None, 'vh_path': None, 'status': 'failed', 'error': str(exc)})
processing_log_df = pd.DataFrame(log_rows)
processing_log_df.to_csv(REPORT_DIR / 'sar_preprocessing_log.csv', index=False)
display(processing_log_df)


## Section 2.4 — SAR QC Visualization

Side-by-side VV and VH plots help verify that the preprocessed backscatter looks physically plausible before the pipeline moves on. This is the quickest way to catch clipping, missing polarization bands, or bad terrain-correction outputs.


In [ ]:
transformer = Transformer.from_crs('EPSG:4326', EPSG_TARGET, always_xy=True)
city_x, city_y = transformer.transform(91.872, 24.899)
for _, row in processing_log_df.query("status == 'success'").iterrows():
    with rasterio.open(row['vv_path']) as vv_src, rasterio.open(row['vh_path']) as vh_src:
        vv = vv_src.read(1)
        vh = vh_src.read(1)
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
        im0 = axes[0].imshow(vv, cmap='gray', vmin=-25, vmax=0)
        axes[0].set_title(f"VV {row['date']}")
        plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
        im1 = axes[1].imshow(vh, cmap='gray', vmin=-25, vmax=0)
        axes[1].set_title(f"VH {row['date']}")
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"sar_qc_{row['date']}.png", dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)


## Section 2.5 — Terrain Layer Computation (for HTC Branch)

Slope, TWI, permanent water, and HAND are the four channels that feed the Haor Terrain Conditioning branch. They are all normalized, reprojected, and resampled to match the SAR grid exactly so the model sees aligned inputs.


In [ ]:
import richdem as rd
import whitebox
from pysheds.grid import Grid

def normalize(arr, clip_max=None):
    arr = np.asarray(arr, dtype=np.float32)
    if clip_max is not None:
        arr = np.clip(arr, 0, clip_max)
    finite = np.isfinite(arr)
    out = np.zeros_like(arr, dtype=np.float32)
    if finite.any():
        mn = arr[finite].min()
        mx = arr[finite].max()
        if mx > mn:
            out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def resample_to_reference(source_path, reference_path, output_path, resampling=Resampling.bilinear, threshold=None):
    with rasterio.open(reference_path) as ref:
        profile = ref.profile.copy()
        destination = np.zeros((ref.height, ref.width), dtype=np.float32)
        with rasterio.open(source_path) as src:
            reproject(source=rasterio.band(src, 1), destination=destination, src_transform=src.transform, src_crs=src.crs, dst_transform=ref.transform, dst_crs=ref.crs, resampling=resampling)
        if threshold is not None:
            destination = (destination > threshold).astype(np.float32)
        profile.update(count=1, dtype='float32')
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(destination.astype(np.float32), 1)

reference_sar = Path(processing_log_df.query("status == 'success'").iloc[0]['vv_path'])
dem_path = RAW_DEM / 'srtm_sylhet_30m.tif'

slope_rd = rd.TerrainAttribute(rd.LoadGDAL(str(dem_path)), attrib='slope_riserun')
slope_raw = RAW_DEM / 'slope_raw.tif'
rd.SaveGDAL(str(slope_raw), slope_rd)

grid = Grid.from_raster(str(dem_path))
dem_array = grid.read_raster(str(dem_path))
pit_filled = grid.fill_pits(dem_array)
flooded = grid.fill_depressions(pit_filled)
inflated = grid.resolve_flats(flooded)
fdir = grid.flowdir(inflated)
acc = grid.accumulation(fdir)
slope_rad = np.arctan(np.clip(np.array(slope_rd, dtype=np.float32), 1e-3, None))
twi = np.log((np.array(acc, dtype=np.float32) + 1.0) / slope_rad)
twi_raw = RAW_DEM / 'twi_raw.tif'
with rasterio.open(dem_path) as dem_src:
    profile = dem_src.profile.copy()
    profile.update(count=1, dtype='float32')
    with rasterio.open(twi_raw, 'w', **profile) as dst:
        dst.write(twi.astype(np.float32), 1)

stream_mask_path = RAW_DEM / 'streams_mask.tif'
with rasterio.open(dem_path) as dem_src:
    profile = dem_src.profile.copy()
    profile.update(count=1, dtype='uint8')
    with rasterio.open(stream_mask_path, 'w', **profile) as dst:
        dst.write((np.array(acc) > np.nanpercentile(np.array(acc), 85)).astype(np.uint8), 1)

hand_raw = RAW_DEM / 'hand_raw.tif'
wbt = whitebox.WhiteboxTools()
wbt.verbose = False
wbt.elevation_above_stream(dem=str(dem_path), streams=str(stream_mask_path), output=str(hand_raw))

resample_to_reference(slope_raw, reference_sar, PROCESSED_TERRAIN / 'slope_sylhet.tif')
resample_to_reference(twi_raw, reference_sar, PROCESSED_TERRAIN / 'twi_sylhet.tif')
resample_to_reference(RAW_DEM / 'jrc_permanent_water.tif', reference_sar, PROCESSED_TERRAIN / 'jrc_water_sylhet.tif', resampling=Resampling.nearest, threshold=90)
resample_to_reference(hand_raw, reference_sar, PROCESSED_TERRAIN / 'hand_sylhet.tif')

for path_out, clip_max in [(PROCESSED_TERRAIN / 'slope_sylhet.tif', None), (PROCESSED_TERRAIN / 'twi_sylhet.tif', None), (PROCESSED_TERRAIN / 'hand_sylhet.tif', 30)]:
    with rasterio.open(path_out) as src:
        arr = normalize(src.read(1), clip_max=clip_max)
        profile = src.profile.copy()
    with rasterio.open(path_out, 'w', **profile) as dst:
        dst.write(arr.astype(np.float32), 1)


## Section 2.6 — Terrain QC Visualization

Plotting all four terrain layers together is the fastest way to verify that reprojection and normalization behaved correctly. It also gives a clear visual summary of why CASA-Net needs topographic context in Sylhet’s haor landscape.


In [ ]:
admin_candidates = list(RAW_INFRA.rglob('*.shp'))
admin_gdf = gpd.read_file(admin_candidates[0]).to_crs(EPSG_TARGET) if admin_candidates else None
terrain_paths = {
    'Slope': PROCESSED_TERRAIN / 'slope_sylhet.tif',
    'TWI': PROCESSED_TERRAIN / 'twi_sylhet.tif',
    'JRC Water': PROCESSED_TERRAIN / 'jrc_water_sylhet.tif',
    'HAND': PROCESSED_TERRAIN / 'hand_sylhet.tif',
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
for ax, (title, path) in zip(axes.ravel(), terrain_paths.items()):
    with rasterio.open(path) as src:
        arr = src.read(1)
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    im = ax.imshow(arr, extent=extent, cmap='viridis')
    if admin_gdf is not None:
        admin_gdf.boundary.plot(ax=ax, color='white', linewidth=0.5)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'terrain_layers_qc.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)


## Section 2.7 — Preprocessing Summary Table

The summary table is the hand-off checkpoint into mask preparation. It confirms which dates produced VV/VH rasters and whether the terrain channels required by HTC are all available.


In [ ]:
summary_rows = []
terrain_ready = {name: path.exists() for name, path in terrain_paths.items()}
for _, row in processing_log_df.iterrows():
    summary_rows.append({'date': row['date'], 'vv_path': row['vv_path'], 'vh_path': row['vh_path'], 'Slope': '✅' if terrain_ready['Slope'] else '❌', 'TWI': '✅' if terrain_ready['TWI'] else '❌', 'JRC': '✅' if terrain_ready['JRC Water'] else '❌', 'HAND': '✅' if terrain_ready['HAND'] else '❌'})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORT_DIR / 'preprocessing_summary.csv', index=False)
display(summary_df)


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 2.8 ? Multi-Hazard Feature Engineering Manifest
Flood uses the current SAR + terrain stack. The multi-hazard upgrade adds planned feature layers for **riverbank erosion** and **landslide susceptibility**, so later notebooks can consume a shared feature manifest instead of a flood-only terrain bundle.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
import pandas as pd
from analysis.multi_hazard_support import load_hazard_catalog, ensure_multihazard_tree

ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')
paths = ensure_multihazard_tree(ROOT, hazard_catalog)

feature_rows = []
feature_templates = {
    'flood': ['slope', 'twi', 'hand', 'jrc_permanent_water'],
    'erosion': ['distance_to_river', 'bankfull_zone', 'yearly_water_change'],
    'landslide': ['slope', 'curvature', 'roughness', 'rainfall_accum_3day', 'rainfall_accum_7day'],
}

for hazard, features in feature_templates.items():
    for feature_name in features:
        output_path = paths['processed_features'] / hazard / f'{feature_name}.tif'
        feature_rows.append({
            'hazard': hazard,
            'feature_name': feature_name,
            'planned_output': str(output_path),
            'exists': output_path.exists(),
        })

multi_hazard_feature_manifest = pd.DataFrame(feature_rows)
multi_hazard_feature_manifest.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_feature_manifest.csv', index=False)
multi_hazard_feature_manifest
